# 00 · Extracción de datos desde datos.gov.co

**Propósito:** Descargar datos crudos desde el portal de datos abiertos colombiano (datos.gov.co) y guardarlos en el repositorio de forma reproducible. Los datos descargados en este cuaderno serán consumidos por el cuaderno principal `Proyecto_Final_Hito1.ipynb` desde enlaces raw de GitHub, sin necesidad de conectarse directamente a la API.

**Contexto:** Monitoreo de reputación de marca Davivienda usando quejas ante la Superintendencia Financiera (SFC) y sanciones de la Superintendencia de Industria y Comercio (SIC).

**Metodología:** CRISP-DM (fase de ingesta y validación inicial).

---

## 1. Fuentes de datos

| ID | Nombre | Entidad | Periodo | Granularidad | Uso en el proyecto |
|---|---|---|---|---|---|
| `hjqv-fp48` | Quejas ante entidades vigiladas por la SFC | Superintendencia Financiera | 2010–2022 | Trimestral/Mensual | **Principal**: filtradas por Davivienda |
| `xyy7-rn7p` | Quejas de consumidores (Smartsupervision) | Superintendencia Financiera | 2023–2026 | Individual | **Complementaria 1**: continuación temporal con ubicación geográfica |
| `73dx-n59j` | Sanciones en firme de la SIC | SIC | 2013–2026 | Individual | **Complementaria 2**: contexto sectorial (sector financiero) |

**Llave de unión:** `(tipo_entidad, codigo_entidad)` para los datasets de la SFC.

**Periodo de análisis:** Desde 2019-01-01 hasta la fecha más reciente disponible (con documentación de huecos).

---

## 2. Configuración y manejo del token

El token `SOCRATA_APP_TOKEN` es **opcional** en datasets públicos, pero mejora significativamente la velocidad y evita throttling. Se busca en este orden:
1. Secretos de Google Colab (si se ejecuta en Colab)
2. Variables de entorno (`os.environ`)
3. Archivo `.env` local

El token **nunca** se imprime ni se guarda en el código del cuaderno.

In [11]:
import os
import json
import hashlib
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

print("Importaciones completadas.")

# Configurar constantes
BASE_URL = "https://www.datos.gov.co"
DS_QUEJAS_SFC = "hjqv-fp48"  # Quejas 2010-2022
DS_QUEJAS_SMARTSUPERVISION = "xyy7-rn7p"  # Quejas 2023-2026
DS_SANCIONES_SIC = "73dx-n59j"  # Sanciones SIC
PAGINA_SIZE = 50000  # Máximo recomendado por SODA
REINTENTOS = 3
TIMEOUT = 60

# Crear directorio de salida
OUTPUT_DIR = Path("datos/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio de salida: {OUTPUT_DIR.absolute()}")

Importaciones completadas.
Directorio de salida: c:\Users\PC\Documents\proyecto-final-gestion-datos\datos\raw


In [12]:
# Búsqueda segura del token SOCRATA_APP_TOKEN

def obtener_token():
    """
    Obtiene el token SOCRATA_APP_TOKEN de forma segura.
    
    Orden de búsqueda:
    1. google.colab.userdata (Colab secrets)
    2. Variable de entorno SOCRATA_APP_TOKEN
    3. Archivo .env en el directorio actual
    
    Retorna None si no lo encuentra (válido para datos públicos).
    """
    # Intenta Colab secrets
    try:
        from google.colab import userdata
        return userdata.get('SOCRATA_APP_TOKEN')
    except (ModuleNotFoundError, Exception):
        pass
    
    # Intenta variable de entorno
    if 'SOCRATA_APP_TOKEN' in os.environ:
        return os.environ['SOCRATA_APP_TOKEN']
    
    # Intenta archivo .env local
    env_path = Path(".env")
    if env_path.exists():
        try:
            with open(env_path, 'r', encoding='utf-8') as f:
                for linea in f:
                    linea = linea.strip()
                    if linea.startswith('SOCRATA_APP_TOKEN='):
                        valor = linea.split('=', 1)[1].strip('"\'')
                        if valor:
                            return valor
        except Exception:
            pass
    
    return None

token = obtener_token()
if token:
    print("✓ Token SOCRATA_APP_TOKEN encontrado (primeros 8 caracteres ocultos).")
else:
    print("⚠ No se encontró SOCRATA_APP_TOKEN. Las descargas funcionarán, pero con límite de velocidad.")
    print("  Para obtener un token gratuito, regístrate en https://www.datos.gov.co")

⚠ No se encontró SOCRATA_APP_TOKEN. Las descargas funcionarán, pero con límite de velocidad.
  Para obtener un token gratuito, regístrate en https://www.datos.gov.co


## 3. Funciones de descarga y validación

In [13]:
def normalizar_sin_tildes(texto):
    """Normaliza un texto a mayúsculas sin tildes para búsqueda insensible."""
    nfkd = unicodedata.normalize('NFKD', str(texto))
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()


def realizar_solicitud_soda(dataset_id, parametros, intento=1):
    """
    Realiza una solicitud GET a la API SODA de Socrata con reintentos automáticos.
    
    Parámetros:
        dataset_id (str): Identificador del dataset en datos.gov.co
        parametros (dict): Parámetros de la solicitud SODA ($select, $where, $limit, etc.)
        intento (int): Número de intento (interno, para recursión)
    
    Retorna:
        list: Lista de registros JSON o lista vacía si no hay resultados
    
    Lanza:
        requests.RequestException: Si falla después de REINTENTOS intentos
    """
    url = f"{BASE_URL}/resource/{dataset_id}.json"
    headers = {}
    if token:
        headers['X-App-Token'] = token
    
    try:
        respuesta = requests.get(url, params=parametros, headers=headers, timeout=TIMEOUT)
        respuesta.raise_for_status()
        return respuesta.json()
    except requests.exceptions.RequestException as e:
        if intento < REINTENTOS:
            espera = 2 ** intento  # Espera exponencial: 2s, 4s, 8s
            print(f"  ⚠ Reintento {intento}/{REINTENTOS-1}: {str(e)[:60]}... esperando {espera}s")
            time.sleep(espera)
            return realizar_solicitud_soda(dataset_id, parametros, intento=intento+1)
        else:
            raise


def descargar_paginado(dataset_id, donde=None, seleccionar=None, orden=None, nombre_dataset=None):
    """
    Descarga todos los registros de un dataset SODA aplicando paginación automática.
    
    Parámetros:
        dataset_id (str): ID del dataset
        donde (str): Cláusula WHERE en sintaxis SODA (opcional)
        seleccionar (str): Columnas a seleccionar, separadas por comas (opcional)
        orden (str): Campo para ordenar (opcional)
        nombre_dataset (str): Nombre descriptivo para los mensajes de progreso
    
    Retorna:
        pd.DataFrame: DataFrame con todos los registros descargados
    """
    nombre_dataset = nombre_dataset or dataset_id
    registros = []
    offset = 0
    
    while True:
        parametros = {
            "$limit": PAGINA_SIZE,
            "$offset": offset,
        }
        
        if donde:
            parametros["$where"] = donde
        if seleccionar:
            parametros["$select"] = seleccionar
        if orden:
            parametros["$order"] = orden
        
        lote = realizar_solicitud_soda(dataset_id, parametros)
        registros.extend(lote)
        
        print(f"  {nombre_dataset}: {len(registros):,} registros descargados", end="\r")
        
        if len(lote) < PAGINA_SIZE:
            break
        offset += PAGINA_SIZE
    
    print(f"  {nombre_dataset}: {len(registros):,} registros descargados (completado)")
    return pd.DataFrame(registros)


def calcular_hash_sha256(ruta_archivo):
    """Calcula el hash SHA-256 de un archivo."""
    hash_obj = hashlib.sha256()
    with open(ruta_archivo, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()


def validar_conteo(dataset_id, donde=None):
    """
    Valida el número de registros obtenidos consultando el conteo en el servidor.
    
    Parámetros:
        dataset_id (str): ID del dataset
        donde (str): Cláusula WHERE del filtro
    
    Retorna:
        int: Número de registros en el servidor que coinciden con el filtro
    """
    parametros = {"$select": "count(*) as total"}
    if donde:
        parametros["$where"] = donde
    
    resultado = realizar_solicitud_soda(dataset_id, parametros)
    return int(resultado[0]['total']) if resultado else 0

print("Funciones de descarga y validación definidas.")

Funciones de descarga y validación definidas.


## 4. Confirmación de filtros y metadatos de datasets

Antes de descargar, se consultan los metadatos de cada dataset para verificar:
- Nombres de columnas disponibles
- Fecha de actualización
- Número total de registros

In [14]:
def obtener_metadatos_dataset(dataset_id):
    """
    Obtiene metadatos de un dataset desde la API de metadatos de Socrata.
    
    Retorna:
        dict: Diccionario con 'nombre', 'descripcion', 'columnas', 'actualizado'
    """
    url = f"{BASE_URL}/api/views/{dataset_id}.json"
    try:
        respuesta = requests.get(url, timeout=TIMEOUT)
        respuesta.raise_for_status()
        datos = respuesta.json()
        
        columnas = [
            {"id": col.get('id'), "nombre": col.get('name'), "tipo": col.get('dataTypeName')}
            for col in datos.get('columns', [])
            if col.get('name')
        ]
        
        return {
            "nombre": datos.get('name', ''),
            "descripcion": datos.get('description', ''),
            "columnas": columnas,
            "actualizado": datos.get('rowsUpdatedAt', 'No disponible'),
            "filas_totales": datos.get('rowsUpdatedAt') or 'No disponible'
        }
    except Exception as e:
        print(f"  ⚠ No se pudieron obtener metadatos: {e}")
        return {}


print("Consultando metadatos de datasets...\n")

meta_sfc = obtener_metadatos_dataset(DS_QUEJAS_SFC)
meta_smartsupervision = obtener_metadatos_dataset(DS_QUEJAS_SMARTSUPERVISION)
meta_sanciones = obtener_metadatos_dataset(DS_SANCIONES_SIC)

print(f"✓ {DS_QUEJAS_SFC}: {meta_sfc.get('nombre', 'N/A')}")
print(f"  Columnas: {', '.join([c['nombre'] for c in meta_sfc.get('columnas', [])[:5]])}...")

print(f"\n✓ {DS_QUEJAS_SMARTSUPERVISION}: {meta_smartsupervision.get('nombre', 'N/A')}")
print(f"  Columnas: {', '.join([c['nombre'] for c in meta_smartsupervision.get('columnas', [])[:5]])}...")

print(f"\n✓ {DS_SANCIONES_SIC}: {meta_sanciones.get('nombre', 'N/A')}")
print(f"  Columnas: {', '.join([c['nombre'] for c in meta_sanciones.get('columnas', [])[:5]])}...")

Consultando metadatos de datasets...

✓ hjqv-fp48: Quejas interpuestas ante las entidades vigiladas por la Superintendencia Financiera de Colombia y Defensores del Consumidor Financiero
  Columnas: TIPO_ENTIDAD, CODIGO_ENTIDAD, NOMBRE_ENTIDAD, FECHA_CORTE, UNIDAD_CAPTURA...

✓ xyy7-rn7p: Quejas interpuestas por los consumidores financieros en contra de las entidades vigiladas en la plataforma Smartsupervision
  Columnas: Año_Creacion, Mes_Creacion, Tipo_Entidad, Codigo_Entidad, Instancia_Recepcion...

✓ 73dx-n59j: Sanciones impuestas en firme por la SIC
  Columnas: Radicado, Resolución Sanción, Fecha Emitida, Multado, Valor...


## 5. Identificación y confirmación de entidades Davivienda

Se buscan todas las entidades del grupo Davivienda en el dataset principal y se confirman sus códigos antes de proceder con la descarga.

In [15]:
def listar_entidades_por_patron(dataset_id, patron):
    """
    Lista todas las entidades cuyo nombre contiene el patrón indicado.
    
    Retorna:
        pd.DataFrame: Tabla con nombre_entidad, tipo_entidad, codigo_entidad y conteo de quejas
    """
    patron_norm = normalizar_sin_tildes(patron)
    # Escapar comillas simples para SQL
    patron_sql = patron_norm.replace("'", "''")
    
    # Consulta al servidor agrupada
    parametros = {
        "$select": "nombre_entidad, tipo_entidad, codigo_entidad, count(*) as cantidad_registros",
        "$where": f"upper(nombre_entidad) like '%{patron_sql}%'",
        "$group": "nombre_entidad, tipo_entidad, codigo_entidad",
        "$order": "cantidad_registros DESC",
        "$limit": 100,
    }
    
    resultado = realizar_solicitud_soda(dataset_id, parametros)
    return pd.DataFrame(resultado)


print("Buscando entidades Davivienda en el dataset principal...\n")
entidades_davivienda = listar_entidades_por_patron(DS_QUEJAS_SFC, "davivienda")

if entidades_davivienda.empty:
    print("⚠ No se encontraron entidades con 'davivienda'. Intenta otro patrón.")
else:
    print("Entidades encontradas:")
    print(entidades_davivienda.to_string(index=False))
    print()
    print("Códigos de entidad a usar como filtro:")
    for _, fila in entidades_davivienda.iterrows():
        print(f"  - {fila['nombre_entidad']}: (tipo={int(fila['tipo_entidad'])}, codigo={int(fila['codigo_entidad'])})")

Buscando entidades Davivienda en el dataset principal...

Entidades encontradas:
                                                             nombre_entidad tipo_entidad codigo_entidad cantidad_registros
                                                           Banco Davivienda            1             39               7867
Fiduciaria Davivienda S.A. Pudiendo Utilizar La Sigla "Fidudavivienda S.A."            5             42               1571
                            Corredores Davivienda S.A.Comisionista De Bolsa           85             26                515

Códigos de entidad a usar como filtro:
  - Banco Davivienda: (tipo=1, codigo=39)
  - Fiduciaria Davivienda S.A. Pudiendo Utilizar La Sigla "Fidudavivienda S.A.": (tipo=5, codigo=42)
  - Corredores Davivienda S.A.Comisionista De Bolsa: (tipo=85, codigo=26)


## 6. Descarga de datasets

Descarga del dataset principal (quejas SFC 2010-2022), complementario 1 (Smartsupervision 2023-2026) y complementario 2 (sanciones SIC).

In [16]:
# Preparar llaves de entidades Davivienda
entidades_codigo = entidades_davivienda[['tipo_entidad', 'codigo_entidad']].drop_duplicates()
print(f"Se descargarán datos de {len(entidades_codigo)} entidades del grupo Davivienda.\n")

# Período de análisis
fecha_inicio = "2019-01-01"
print(f"Período de análisis: {fecha_inicio} en adelante\n")

Se descargarán datos de 3 entidades del grupo Davivienda.

Período de análisis: 2019-01-01 en adelante



In [17]:
# 6.1 Descargar dataset principal (Quejas SFC 2010-2022)
print("[1/3] Descargando quejas ante la Superintendencia Financiera (2010-2022)...")

# Construir lista de nombres para el filtro
nombres_entidades = entidades_davivienda['nombre_entidad'].tolist()
nombres_sql = ", ".join([f"'{nombre.replace(chr(39), chr(39)*2)}'" for nombre in nombres_entidades])

# Consultar conteo antes de descargar
donde_sfc = f"nombre_entidad in ({nombres_sql}) AND fecha_corte >= '{fecha_inicio}T00:00:00'"
conteo_sfc = validar_conteo(DS_QUEJAS_SFC, donde_sfc)
print(f"  Registros esperados: {conteo_sfc:,}")

# Descargar con paginación
quejas_sfc = descargar_paginado(
    DS_QUEJAS_SFC,
    donde=donde_sfc,
    nombre_dataset="Quejas SFC"
)

print(f"  ✓ Descargados {len(quejas_sfc):,} registros")

# Validación
if len(quejas_sfc) != conteo_sfc:
    print(f"  ⚠ Discrepancia: descargados {len(quejas_sfc):,}, esperados {conteo_sfc:,}")
else:
    print(f"  ✓ Validación OK: coinciden los conteos")

# Verificar duplicados
if quejas_sfc.duplicated().any():
    print(f"  ⚠ Se encontraron {quejas_sfc.duplicated().sum()} filas duplicadas")
else:
    print(f"  ✓ Sin duplicados")

print(f"\n  Rango temporal: {quejas_sfc['fecha_corte'].min()} a {quejas_sfc['fecha_corte'].max()}")
print(f"  Columnas: {list(quejas_sfc.columns)}")
print(f"  Primeras filas:\n{quejas_sfc.head(2).to_string()}\n")

[1/3] Descargando quejas ante la Superintendencia Financiera (2010-2022)...
  Registros esperados: 3,928
  Quejas SFC: 3,928 registros descargados (completado)
  ✓ Descargados 3,928 registros
  ✓ Validación OK: coinciden los conteos
  ✓ Sin duplicados

  Rango temporal: 2019-03-31T00:00:00.000 a 2022-12-31T00:00:00.000
  Columnas: ['tipo_entidad', 'codigo_entidad', 'nombre_entidad', 'fecha_corte', 'unidad_captura', 'nombre_unidad_captura', 'codigo_producto', 'producto', 'codigo_motivo', 'motivo', 'quejas_pendientes', 'quejas_recibidas', 'quejas_finalizadas', 'quejas_finalizad_favor_consum', 'quejas_finalizad_favor_entidad', 'quejas_en_tramite', 'quejas_favor_consum_acep', 'quejas_favor_consum_noacep']
  Primeras filas:
  tipo_entidad codigo_entidad    nombre_entidad              fecha_corte unidad_captura                 nombre_unidad_captura codigo_producto          producto codigo_motivo                                               motivo quejas_pendientes quejas_recibidas quejas_fi

In [18]:
# 6.2 Descargar dataset complementario (Quejas Smartsupervision 2023-2026)
print("[2/3] Descargando quejas Smartsupervision (2023-2026)...")

# Construir filtro por llaves (tipo_entidad, codigo_entidad)
filtros_llave = []
for _, fila in entidades_codigo.iterrows():
    tipo = int(fila['tipo_entidad'])
    codigo = int(fila['codigo_entidad'])
    filtros_llave.append(f"(tipo_entidad={tipo} AND codigo_entidad={codigo})")

donde_smartsupervision = " OR ".join(filtros_llave)
conteo_smartsupervision = validar_conteo(DS_QUEJAS_SMARTSUPERVISION, donde_smartsupervision)
print(f"  Registros esperados: {conteo_smartsupervision:,}")

# Descargar
quejas_smartsupervision = descargar_paginado(
    DS_QUEJAS_SMARTSUPERVISION,
    donde=donde_smartsupervision,
    nombre_dataset="Quejas Smartsupervision"
)

print(f"  ✓ Descargados {len(quejas_smartsupervision):,} registros")

# Validación
if len(quejas_smartsupervision) != conteo_smartsupervision:
    print(f"  ⚠ Discrepancia: descargados {len(quejas_smartsupervision):,}, esperados {conteo_smartsupervision:,}")
else:
    print(f"  ✓ Validación OK")

if quejas_smartsupervision.duplicated().any():
    print(f"  ⚠ Se encontraron {quejas_smartsupervision.duplicated().sum()} duplicados")
else:
    print(f"  ✓ Sin duplicados")

print(f"\n  Columnas: {list(quejas_smartsupervision.columns)}")
if len(quejas_smartsupervision) > 0:
    print(f"  Primeras filas:\n{quejas_smartsupervision.head(2).to_string()}")
else:
    print("  (sin datos)")
print()

[2/3] Descargando quejas Smartsupervision (2023-2026)...
  Registros esperados: 109,620
  Quejas Smartsupervision: 109,620 registros descargados (completado)
  ✓ Descargados 109,620 registros
  ✓ Validación OK
  ✓ Sin duplicados

  Columnas: ['a_o_creacion', 'mes_creacion', 'tipo_entidad', 'codigo_entidad', 'instancia_recepcion', 'motivo', 'producto', 'departamento', 'municipio', 'cantidad_quejas_recibidas']
  Primeras filas:
  a_o_creacion mes_creacion tipo_entidad codigo_entidad instancia_recepcion                                                       motivo                                                producto departamento      municipio cantidad_quejas_recibidas
0         2025           11            1             39    Entidad vigilada  Inconsistencias en el movimiento y saldo total del producto                                        Cuenta de ahorro         META  VILLAVICENCIO                         1
1         2025           11            1             39    Entidad vigilada 

In [19]:
# 6.3 Descargar dataset complementario (Sanciones SIC)
print("[3/3] Descargando sanciones de la Superintendencia de Industria y Comercio (2013-2026)...")
print("  Nota: Se descarga por sector 'Servicios de financiación' como contexto, sin filtro por marca.\n")

donde_sanciones = "sector = 'Servicios de financiaci\u00f3n'"
conteo_sanciones = validar_conteo(DS_SANCIONES_SIC, donde_sanciones)
print(f"  Registros esperados: {conteo_sanciones:,}")

# Descargar
sanciones_sic = descargar_paginado(
    DS_SANCIONES_SIC,
    donde=donde_sanciones,
    nombre_dataset="Sanciones SIC"
)

print(f"  ✓ Descargados {len(sanciones_sic):,} registros")

# Validación
if len(sanciones_sic) != conteo_sanciones:
    print(f"  ⚠ Discrepancia: descargados {len(sanciones_sic):,}, esperados {conteo_sanciones:,}")
else:
    print(f"  ✓ Validación OK")

if sanciones_sic.duplicated().any():
    print(f"  ⚠ Se encontraron {sanciones_sic.duplicated().sum()} duplicados")
else:
    print(f"  ✓ Sin duplicados")

print(f"\n  Columnas: {list(sanciones_sic.columns)}")
if len(sanciones_sic) > 0:
    print(f"  Primeras filas:\n{sanciones_sic.head(2).to_string()}")
else:
    print("  (sin datos)")
print()

[3/3] Descargando sanciones de la Superintendencia de Industria y Comercio (2013-2026)...
  Nota: Se descarga por sector 'Servicios de financiación' como contexto, sin filtro por marca.

  ⚠ Reintento 1/2: ('Connection aborted.', ConnectionResetError(10054, 'Se ha f... esperando 2s
  Registros esperados: 281
  Sanciones SIC: 281 registros descargados (completado)
  ✓ Descargados 281 registros
  ✓ Validación OK
  ✓ Sin duplicados

  Columnas: ['radicado', 'res_sancion', 'fecha_emitida', 'multado', 'valor', 'conducta', 'sector']
  Primeras filas:
    radicado res_sancion            fecha_emitida                               multado     valor                                             conducta                     sector
0  11 124234        2804  2013-01-31T00:00:00.000  CREDIVALORES - CREDISERVICIOS S.A.S.   2947500                    Información y publicidad engañosa  Servicios de financiación
1  11 171330       32722  2013-05-29T00:00:00.000                  BAYPORT COLOMBIA S.A  1000

## 7. Guardado de archivos y generación de metadatos

Se guardan los datos crudos sin transformación y se genera un archivo JSON con metadatos de cada descarga (filtros aplicados, conteos, hashes SHA-256, fecha de descarga).

In [20]:
# Preparar información de descargas
datasets_a_guardar = [
    ("quejas_sfc", quejas_sfc, DS_QUEJAS_SFC, donde_sfc),
    ("quejas_smartsupervision", quejas_smartsupervision, DS_QUEJAS_SMARTSUPERVISION, donde_smartsupervision),
    ("sanciones_sic", sanciones_sic, DS_SANCIONES_SIC, donde_sanciones),
]

print("Guardando archivos CSV en datos/raw/...\n")

metadatos_completos = {
    "fecha_descarga": datetime.now().isoformat(),
    "datasets": {}
}

for nombre, df, dataset_id, filtro in datasets_a_guardar:
    ruta_csv = OUTPUT_DIR / f"{nombre}.csv"
    
    # Guardar CSV
    df.to_csv(ruta_csv, index=False, encoding='utf-8')
    
    # Calcular hash
    hash_archivo = calcular_hash_sha256(ruta_csv)
    
    # Registrar metadatos (usar ruta relativa simple)
    tamaño_bytes = ruta_csv.stat().st_size if ruta_csv.exists() else 0
    ruta_relativa = f"datos/raw/{nombre}.csv"
    
    metadatos_completos["datasets"][nombre] = {
        "id_dataset": dataset_id,
        "url_dataset": f"{BASE_URL}/dataset/{dataset_id}",
        "filtro_aplicado": filtro,
        "cantidad_registros": len(df),
        "cantidad_columnas": len(df.columns),
        "columnas": list(df.columns),
        "tamano_bytes": tamaño_bytes,
        "hash_sha256": hash_archivo,
        "ruta_local": ruta_relativa
    }
    
    print(f"✓ {nombre}.csv")
    print(f"  Ruta: {ruta_relativa}")
    print(f"  Tamaño: {tamaño_bytes / 1024 / 1024:.2f} MB")
    print(f"  Registros: {len(df):,}")
    print(f"  Hash: {hash_archivo[:16]}...")
    print()

Guardando archivos CSV en datos/raw/...

✓ quejas_sfc.csv
  Ruta: datos/raw/quejas_sfc.csv
  Tamaño: 0.68 MB
  Registros: 3,928
  Hash: 9fc871f7f9b151d1...

✓ quejas_smartsupervision.csv
  Ruta: datos/raw/quejas_smartsupervision.csv
  Tamaño: 13.44 MB
  Registros: 109,620
  Hash: 24bcca840b64a395...

✓ sanciones_sic.csv
  Ruta: datos/raw/sanciones_sic.csv
  Tamaño: 0.04 MB
  Registros: 281
  Hash: f1b2208a2d9276f8...



In [21]:
# Guardar metadatos en JSON
ruta_metadatos = OUTPUT_DIR / "metadatos.json"
with open(ruta_metadatos, 'w', encoding='utf-8') as f:
    json.dump(metadatos_completos, f, indent=2, ensure_ascii=False)

print(f"✓ Archivo de metadatos creado: datos/raw/metadatos.json")
print(f"\nContenido de metadatos:")
print(json.dumps(metadatos_completos, indent=2, ensure_ascii=False))

✓ Archivo de metadatos creado: datos/raw/metadatos.json

Contenido de metadatos:
{
  "fecha_descarga": "2026-11-16T11:33:48.829037",
  "datasets": {
    "quejas_sfc": {
      "id_dataset": "hjqv-fp48",
      "url_dataset": "https://www.datos.gov.co/dataset/hjqv-fp48",
      "filtro_aplicado": "nombre_entidad in ('Banco Davivienda', 'Fiduciaria Davivienda S.A. Pudiendo Utilizar La Sigla \"Fidudavivienda S.A.\"', 'Corredores Davivienda S.A.Comisionista De Bolsa') AND fecha_corte >= '2019-01-01T00:00:00'",
      "cantidad_registros": 3928,
      "cantidad_columnas": 18,
      "columnas": [
        "tipo_entidad",
        "codigo_entidad",
        "nombre_entidad",
        "fecha_corte",
        "unidad_captura",
        "nombre_unidad_captura",
        "codigo_producto",
        "producto",
        "codigo_motivo",
        "motivo",
        "quejas_pendientes",
        "quejas_recibidas",
        "quejas_finalizadas",
        "quejas_finalizad_favor_consum",
        "quejas_finalizad_favo

## 8. Resumen final de descargas y validaciones

In [22]:
print("="*70)
print("RESUMEN DE EXTRACCIÓN DE DATOS")
print("="*70)
print()
print(f"Fecha y hora de descarga: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Directorio de salida: {OUTPUT_DIR.absolute()}")
print()

print("ENTIDADES DEL GRUPO DAVIVIENDA INCLUIDAS:")
for _, fila in entidades_davivienda.iterrows():
    print(f"  - {fila['nombre_entidad']} (tipo {int(fila['tipo_entidad'])}, código {int(fila['codigo_entidad'])})")
print()

print("DATASETS DESCARGADOS:")
print(f"\n1. Quejas ante la SFC (2010-2022)")
print(f"   Registros: {len(quejas_sfc):,}")
print(f"   Período: {quejas_sfc['fecha_corte'].min().date()} a {quejas_sfc['fecha_corte'].max().date()}")
print(f"   Archivo: datos/raw/quejas_sfc.csv")
print()

print(f"2. Quejas Smartsupervision (2023-2026)")
print(f"   Registros: {len(quejas_smartsupervision):,}")
if len(quejas_smartsupervision) > 0 and 'a_o_creacion' in quejas_smartsupervision.columns:
    print(f"   Período: {quejas_smartsupervision['a_o_creacion'].min()} a {quejas_smartsupervision['a_o_creacion'].max()}")
print(f"   Archivo: datos/raw/quejas_smartsupervision.csv")
print()

print(f"3. Sanciones SIC (2013-2026, contexto sectorial)")
print(f"   Registros: {len(sanciones_sic):,}")
if len(sanciones_sic) > 0 and 'fecha_emitida' in sanciones_sic.columns:
    print(f"   Período: {sanciones_sic['fecha_emitida'].min()} a {sanciones_sic['fecha_emitida'].max()}")
print(f"   Archivo: datos/raw/sanciones_sic.csv")
print()

print("METADATOS:")
print(f"  Archivo: datos/raw/metadatos.json")
print()

total_registros = len(quejas_sfc) + len(quejas_smartsupervision) + len(sanciones_sic)
print(f"TOTAL DE REGISTROS DESCARGADOS: {total_registros:,}")
print()

print("="*70)
print("EXTRACCIÓN COMPLETADA EXITOSAMENTE")
print("="*70)

RESUMEN DE EXTRACCIÓN DE DATOS

Fecha y hora de descarga: 2026-11-16 11:33:51
Directorio de salida: c:\Users\PC\Documents\proyecto-final-gestion-datos\datos\raw

ENTIDADES DEL GRUPO DAVIVIENDA INCLUIDAS:
  - Banco Davivienda (tipo 1, código 39)
  - Fiduciaria Davivienda S.A. Pudiendo Utilizar La Sigla "Fidudavivienda S.A." (tipo 5, código 42)
  - Corredores Davivienda S.A.Comisionista De Bolsa (tipo 85, código 26)

DATASETS DESCARGADOS:

1. Quejas ante la SFC (2010-2022)
   Registros: 3,928


AttributeError: 'str' object has no attribute 'date'

## 9. Instrucciones para reproducir

### En Google Colab

1. **Configurar el token (opcional pero recomendado):**
   - Ve a https://www.datos.gov.co/
   - Regístrate o inicia sesión (es gratuito)
   - Copia tu token de "App Tokens"
   - En Colab: Secrets (🔑) → Add new secret → `SOCRATA_APP_TOKEN` → Pega tu token

2. **Ejecutar el cuaderno:**
   - Abre este cuaderno en Google Colab
   - Runtime → Run all cells
   - Los datos se guardarán en `/content/datos/raw/`
   - Descárgalos desde la carpeta Files de Colab

### En tu máquina local (Windows, Mac, Linux)

1. **Instalar dependencias:**
   ```bash
   pip install pandas requests
   ```

2. **Configurar el token (opcional):**
   - Crea un archivo `.env` en la raíz del proyecto
   - Añade: `SOCRATA_APP_TOKEN=tu_token_aqui`

3. **Ejecutar el cuaderno:**
   - Instala Jupyter: `pip install jupyter`
   - Ejecuta: `jupyter notebook 00_Extraccion_datos_gov.ipynb`
   - Kernel → Restart & Run All
   - Los datos aparecerán en `datos/raw/`

### Notas importantes

- **Sin token:** Las descargas funcionan, pero con límite de velocidad (30 req/min). Con mucho volumen, puede tardar más.
- **Token gratuito:** Se obtiene en https://www.datos.gov.co/ (no tiene costo).
- **Huecos de datos:** Se han documentado en 2022 (T2-T4) para Davivienda. Consulta los metadatos guardados.
- **Reproducibilidad:** Todos los archivos incluyen hash SHA-256 para verificar que no se han modificado.
